# 02 - Verificacion de Eventos en Kafka

Este notebook verifica que los eventos del producer esten llegando correctamente a Kafka.

## 1. Instalacion de kafka-python

In [1]:
import sys
!{sys.executable} -m pip install kafka-python-ng pandas python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 4.5 MB/s eta 0:00:00a 0:00:01


## 2. Consumo de mensajes de Kafka

In [ ]:
from kafka import KafkaConsumer
import json

KAFKA_BROKER = "kafka:29092"
TOPIC = "iot.air_quality.streaming"

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="earliest",
    consumer_timeout_ms=10000,
    value_deserializer=lambda m: json.loads(m.decode("utf-8"))
)

print(f"Consumiendo mensajes del topico: {TOPIC}")
print("Esperando 10 segundos...\n")

messages = []
for msg in consumer:
    messages.append(msg.value)
    print(f"[{len(messages)}] Estacion: {msg.value.get('estacion')} | Temp: {msg.value.get('temperatura')} | IAQ: {msg.value.get('iaq')}")

print(f"\nTotal mensajes recibidos: {len(messages)}")
consumer.close()

Consumiendo mensajes del topico: iot.air_quality.streaming
Esperando 10 segundos...

[1] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 39
[2] Estacion: ESP32_02 | Temp: 25.1 | IAQ: 39
[3] Estacion: ESP32_03 | Temp: 25.52 | IAQ: 47
[4] Estacion: ESP32_04 | Temp: 25.46 | IAQ: 62
[5] Estacion: ESP32_05 | Temp: 25.42 | IAQ: 39
[6] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 40
[7] Estacion: ESP32_02 | Temp: 25.53 | IAQ: 44
[8] Estacion: ESP32_03 | Temp: 26.6 | IAQ: 45
[9] Estacion: ESP32_04 | Temp: 25.6 | IAQ: 67
[10] Estacion: ESP32_05 | Temp: 25.46 | IAQ: 38
[11] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 40
[12] Estacion: ESP32_02 | Temp: 25.59 | IAQ: 38
[13] Estacion: ESP32_03 | Temp: 25.72 | IAQ: 46
[14] Estacion: ESP32_04 | Temp: 25.5 | IAQ: 59
[15] Estacion: ESP32_05 | Temp: 25.58 | IAQ: 47
[16] Estacion: ESP32_01 | Temp: 25.4 | IAQ: 46
[17] Estacion: ESP32_02 | Temp: 25.08 | IAQ: 41
[18] Estacion: ESP32_03 | Temp: 25.92 | IAQ: 41
[19] Estacion: ESP32_04 | Temp: 25.43 | IAQ: 60
[20] Estacion: ESP32

## 3. Analisis de mensajes recibidos

In [3]:
import pandas as pd

if messages:
    df = pd.DataFrame(messages)
    print("\nEstructura del evento:")
    print(df.columns.tolist())
    print("\nPrimer evento completo:")
    print(df.iloc[0].to_dict())
    print("\nEstaciones detectadas:")
    print(df["estacion"].value_counts())
else:
    print("No se recibieron mensajes. Verifica que el producer este corriendo.")


Estructura del evento:
['estacion', 'temperatura', 'humedad', 'presion', 'altura', 'gas', 'iaq', 'eco2', 'VOC', 'calidad_aire', 'created_at', 'event_timestamp', 'delayed']

Primer evento completo:
{'estacion': 'ESP32_01', 'temperatura': 25.4, 'humedad': 58.0, 'presion': 1014.08, 'altura': 3820, 'gas': 116, 'iaq': 39, 'eco2': 643, 'VOC': 0.366, 'calidad_aire': 'BUENA', 'created_at': '2026-05-19T01:01:24.869305', 'event_timestamp': 1779152484869, 'delayed': nan}

Estaciones detectadas:
estacion
ESP32_01    108
ESP32_02    108
ESP32_03    108
ESP32_04    108
ESP32_05    108
Name: count, dtype: int64


## 4. Verificacion de eventos retrasados (ESP32_05)

In [4]:
if messages:
    delayed = [m for m in messages if m.get("delayed", False)]
    print(f"Eventos retrasados detectados: {len(delayed)}")
    if delayed:
        print("\nEjemplo de evento retrasado:")
        print(f"  created_at: {delayed[0].get('created_at')}")
        print(f"  estacion: {delayed[0].get('estacion')}")
        print(f"  delayed: {delayed[0].get('delayed')}")

Eventos retrasados detectados: 108

Ejemplo de evento retrasado:
  created_at: 2026-05-19T01:01:20.332814
  estacion: ESP32_05
  delayed: True


## 5. Verificacion de latencia

In [5]:
from datetime import datetime

if messages:
    latencias = []
    now = datetime.utcnow().timestamp() * 1000
    for m in messages:
        if m.get("event_timestamp"):
            lat = now - m["event_timestamp"]
            latencias.append(lat)
    
    if latencias:
        print(f"Latencia promedio: {sum(latencias)/len(latencias):.0f} ms")
        print(f"Latencia minima: {min(latencias):.0f} ms")
        print(f"Latencia maxima: {max(latencias):.0f} ms")

Latencia promedio: 69849 ms
Latencia minima: 15196 ms
Latencia maxima: 127540 ms
